In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = r"D:\metropt+3+dataset\MetroPT3(AirCompressor).csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1516948, 17)

Columns:
['Unnamed: 0', 'timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']


In [2]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp").reset_index(drop=True)

print("Start time:", df["timestamp"].min())
print("End time:", df["timestamp"].max())

print("\nFirst 5 timestamps:")
print(df["timestamp"].head())

Start time: 2020-02-01 00:00:00
End time: 2020-09-01 03:59:50

First 5 timestamps:
0   2020-02-01 00:00:00
1   2020-02-01 00:00:10
2   2020-02-01 00:00:19
3   2020-02-01 00:00:29
4   2020-02-01 00:00:39
Name: timestamp, dtype: datetime64[us]


In [3]:
failure_periods = [
    {
        "failure_id": 1,
        "start": "2020-04-18 00:00:00",
        "end": "2020-04-18 23:59:00"
    },
    {
        "failure_id": 2,
        "start": "2020-05-29 23:30:00",
        "end": "2020-05-30 06:00:00"
    },
    {
        "failure_id": 3,
        "start": "2020-06-05 10:00:00",
        "end": "2020-06-07 14:30:00"
    },
    {
        "failure_id": 4,
        "start": "2020-07-15 14:30:00",
        "end": "2020-07-15 19:00:00"
    }
]

failure_df = pd.DataFrame(failure_periods)

failure_df["start"] = pd.to_datetime(failure_df["start"])
failure_df["end"] = pd.to_datetime(failure_df["end"])

print(failure_df)


   failure_id               start                 end
0           1 2020-04-18 00:00:00 2020-04-18 23:59:00
1           2 2020-05-29 23:30:00 2020-05-30 06:00:00
2           3 2020-06-05 10:00:00 2020-06-07 14:30:00
3           4 2020-07-15 14:30:00 2020-07-15 19:00:00


In [4]:
df["failure"] = 0

for _, failure in failure_df.iterrows():
    mask = (
        (df["timestamp"] >= failure["start"]) &
        (df["timestamp"] <= failure["end"])
    )
    
    df.loc[mask, "failure"] = 1

print("Failure class distribution:")
print(df["failure"].value_counts())

print("\nFailure percentages:")
print((df["failure"].value_counts(normalize=True) * 100).round(3))

Failure class distribution:
failure
0    1486994
1      29954
Name: count, dtype: int64

Failure percentages:
failure
0    98.025
1     1.975
Name: proportion, dtype: float64


In [5]:
failure_data = df[df["failure"] == 1].copy()
normal_data = df[df["failure"] == 0].copy()

# Randomly sample normal observations to match the number of failures
normal_sample = normal_data.sample(
    n=len(failure_data),
    random_state=42
)

balanced_df = pd.concat(
    [failure_data, normal_sample],
    ignore_index=True
)

# Shuffle the resulting dataset
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced dataset shape:", balanced_df.shape)

print("\nClass distribution:")
print(balanced_df["failure"].value_counts())

print("\nClass percentages:")
print(
    (balanced_df["failure"].value_counts(normalize=True) * 100).round(2)
)

Balanced dataset shape: (59908, 18)

Class distribution:
failure
0    29954
1    29954
Name: count, dtype: int64

Class percentages:
failure
0    50.0
1    50.0
Name: proportion, dtype: float64


In [7]:
model_features = [
    "TP2",
    "H1",
    "Motor_current",
    "Oil_temperature",
    "DV_pressure"
]

X = balanced_df[model_features].copy()
y = balanced_df["failure"].copy()

print("Selected features:")
print(model_features)

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 observations:")
print(X.head())

Selected features:
['TP2', 'H1', 'Motor_current', 'Oil_temperature', 'DV_pressure']

Feature matrix shape: (59908, 5)
Target shape: (59908,)

First 5 observations:
     TP2      H1  Motor_current  Oil_temperature  DV_pressure
0  0.004  10.144         4.0975           57.050        0.000
1 -0.010   8.338         0.0350           59.000       -0.014
2  8.954  -0.008         5.6600           74.050        2.016
3  8.348  -0.008         5.6100           75.075        2.214
4 -0.012   9.036         0.0450           65.475       -0.020


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training set: (47926, 5)
Testing set: (11982, 5)

Training class distribution:
failure
0    23963
1    23963
Name: count, dtype: int64

Testing class distribution:
failure
1    5991
0    5991
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

baseline_model.fit(X_train_scaled, y_train)

y_pred = baseline_model.predict(X_test_scaled)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Logistic Regression Performance")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Logistic Regression Performance
--------------------------------
Accuracy : 0.9821
Precision: 0.9722
Recall   : 0.9925
F1-score : 0.9822

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      5991
           1       0.97      0.99      0.98      5991

    accuracy                           0.98     11982
   macro avg       0.98      0.98      0.98     11982
weighted avg       0.98      0.98      0.98     11982


Confusion Matrix:
[[5821  170]
 [  45 5946]]


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest Performance")
print("-------------------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))


Random Forest Performance
-------------------------
Accuracy : 0.9960
Precision: 0.9957
Recall   : 0.9963
F1-score : 0.9960

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5991
           1       1.00      1.00      1.00      5991

    accuracy                           1.00     11982
   macro avg       1.00      1.00      1.00     11982
weighted avg       1.00      1.00      1.00     11982


Confusion Matrix:
[[5965   26]
 [  22 5969]]


In [12]:
baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        rf_accuracy
    ],
    "Precision": [
        precision,
        rf_precision
    ],
    "Recall": [
        recall,
        rf_recall
    ],
    "F1_score": [
        f1,
        rf_f1
    ]
})

baseline_results.round(4)

,Model,Accuracy,Precision,Recall,F1_score
0,Logistic Regression,0.9821,0.9722,0.9925,0.9822
1,Random Forest,0.9960,0.9957,0.9963,0.9960
